In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Maintenance Scheduling

This tutorial demonstrates maintenance scheduling for generators. Setting `maintainable=True` introduces binary variables that schedule contiguous maintenance blocks, reducing or eliminating the component's available capacity during those periods.

To enable maintenance scheduling on a component (`Generator`, `Link` or `Process`), set its attribute `maintainable=True` and specify `maintenance_duration`, the elapsed time per maintenance event in units of the snapshot weightings (hours by default).

In [ ]:
import matplotlib.pyplot as plt

## Basic Maintenance Scheduling

A cheap generator must undergo 3 hours of maintenance (here, 3 snapshots). The optimizer places the maintenance block where the backup generator can cover the load at minimum cost.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=3,
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)

load = [80] * 10
load[0] = 10
load[1] = 10
load[2] = 10
n.add("Load", "load", bus="bus", p_set=load)

In [ ]:
n.optimize()

The maintenance is scheduled during the low-demand period (snapshots 0-2), minimizing the cost of dispatching the expensive backup generator:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Partial Maintenance

With `maintenance_pu=0.5`, only 50% of capacity is unavailable during maintenance. The generator can still dispatch at reduced capacity.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=2,
    maintenance_pu=0.5,
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)
n.add("Load", "load", bus="bus", p_set=80)

In [ ]:
n.optimize()

During maintenance, the cheap generator is limited to 50 MW (50% of 100 MW). The backup covers the remaining 30 MW of the 80 MW load:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Partial Maintenance)")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Multiple Events

With `maintenance_events=2`, the generator must undergo two separate maintenance blocks of `maintenance_duration=2` hours each.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=2,
    maintenance_events=2,
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)
n.add("Load", "load", bus="bus", p_set=50)

In [ ]:
n.optimize()

Both maintenance blocks are contiguous, and the total maintenance duration is 4 hours (2 events x 2 hours):

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Multiple Events)")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Staggered Maintenance for Multiple Generators

When multiple generators require maintenance, the optimizer will stagger their maintenance windows to maintain system adequacy. Here, two generators each need 2-snapshot maintenance, but the load requires at least one to be online at all times.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

for i in range(2):
    n.add(
        "Generator",
        f"gen{i}",
        bus="bus",
        p_nom=100,
        marginal_cost=10,
        maintainable=True,
        maintenance_duration=2,
    )

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)
n.add("Load", "load", bus="bus", p_set=150)

In [ ]:
n.optimize()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Staggered Maintenance)")

n.generators_t.maintenance[["gen0", "gen1"]].plot.bar(ax=axes[1])
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Weighted-Time Duration

`maintenance_duration` is an *elapsed time*, not a snapshot count. Each event covers the minimal run of consecutive snapshots whose `snapshot_weightings` sum to at least the requested duration (rounding up). With non-uniform or coarse snapshots, the number of covered snapshots therefore differs from the duration value.

Here the snapshots represent 3-hour intervals, so a `maintenance_duration=6` hours covers just two snapshots.

In [ ]:
n = pypsa.Network(snapshots=range(8))
n.snapshot_weightings.loc[:, :] = 3  # each snapshot represents 3 elapsed hours

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=6,  # 6 elapsed hours -> 2 snapshots of 3 h each
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)

load = [80] * 8
load[0] = 10
load[1] = 10
n.add("Load", "load", bus="bus", p_set=load)

In [ ]:
n.optimize()

The maintenance status is 1 for two snapshots (6 h / 3 h), even though `maintenance_duration=6`:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Weighted Duration)")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Capacity Expansion

Maintenance scheduling combines with capacity expansion (`p_nom_extendable=True`). The unavailable capacity during maintenance is the bilinear product of the *optimised* capacity and the maintenance status, which PyPSA linearises exactly with a McCormick envelope (an internal auxiliary variable $z = \hat{p} \cdot m$). This requires a finite `p_nom_max`; an infinite one raises a consistency error.

The optimiser co-optimises how much capacity to build and when to maintain it.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom_extendable=True,
    p_nom_max=150,  # must be finite for the McCormick linearisation
    capital_cost=50,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=3,
)

n.add("Generator", "backup", bus="bus", p_nom=200, marginal_cost=100)

load = [100] * 10
load[0] = 20
load[1] = 20
load[2] = 20
n.add("Load", "load", bus="bus", p_set=load)

In [ ]:
n.optimize()

The cheap generator is built at 100 MW and its maintenance is placed in the low-demand window, where it is fully unavailable and the backup covers the residual load:

In [ ]:
print(f"Optimised capacity: {n.generators.p_nom_opt['cheap']:.0f} MW")

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Capacity Expansion)")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Unit Commitment

Maintenance scheduling also combines with unit commitment (`committable=True`). The commitment status $u$ and the maintenance status $m$ are decoupled exactly (via the product $w = u \cdot m$), so a committed unit may be **shut down** while in maintenance rather than being forced to stay online.

Here the generator has a stand-by cost, so it economically prefers to switch off ($u=0$) during its full-outage maintenance window.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    committable=True,
    p_min_pu=0.3,
    stand_by_cost=20,
    maintainable=True,
    maintenance_duration=3,
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)

load = [80] * 10
load[0] = 10
load[1] = 10
load[2] = 10
n.add("Load", "load", bus="bus", p_set=load)

In [ ]:
n.optimize()

The commitment status drops to 0 exactly during the maintenance block, i.e. the unit is shut down while under maintenance instead of idling online:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.status["cheap"].plot.bar(ax=axes[0], color="C0")
axes[0].set_ylabel("Commitment status")
axes[0].set_title("Commitment Status")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

!!! note

    When a minimum up time or start-up cost forces the unit to stay committed, start-up/shut-down dynamics interact with the maintenance events. See the caveats in the [maintenance scheduling user guide](../../user-guide/optimization/maintenance-scheduling.md).

## Links and Processes

The same attributes (`maintainable`, `maintenance_duration`, `maintenance_pu`, `maintenance_events`) apply to [`Link`][pypsa.components.Links] and [`Process`][pypsa.components.Processes] components, where the identical formulation constrains their power flow $p$. The `Process` component follows the same formulation as the `Link`. See the [user guide](../../user-guide/optimization/maintenance-scheduling.md) for the full mathematical description.